# Chapter 1 practice: does money buy happiness?
The book's Example 1-1, made runnable. Run one cell at a time (Shift+Enter). Read the note above each cell first.

Idea: use a country's GDP per capita (input, the feature) to predict its life satisfaction score (output, the label). This is **supervised regression** with a **linear model**.

## 1. Imports and data
The book's data files are public on GitHub, so we load them straight from the link. No download or install needed in Colab.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn.linear_model

base = "https://raw.githubusercontent.com/ageron/handson-ml2/master/datasets/lifesat/"
oecd_bli = pd.read_csv(base + "oecd_bli_2015.csv", thousands=',')
gdp_per_capita = pd.read_csv(base + "gdp_per_capita.csv", thousands=',',
                             delimiter='\t', encoding='latin1', na_values="n/a")
oecd_bli.head()

,LOCATION,Country,INDICATOR,Indicator,MEASURE,Measure,INEQUALITY,Inequality,Unit Code,Unit,PowerCode Code,PowerCode,Reference Period Code,Reference Period,Value,Flag Codes,Flags
0,AUS,Australia,HO_BASE,Dwellings without basic facilities,L,Value,TOT,Total,PC,Percentage,0,units,NaN,NaN,1.1,E,Estimated value
1,AUT,Austria,HO_BASE,Dwellings without basic facilities,L,Value,TOT,Total,PC,Percentage,0,units,NaN,NaN,1.0,NaN,NaN
2,BEL,Belgium,HO_BASE,Dwellings without basic facilities,L,Value,TOT,Total,PC,Percentage,0,units,NaN,NaN,2.0,NaN,NaN
3,CAN,Canada,HO_BASE,Dwellings without basic facilities,L,Value,TOT,Total,PC,Percentage,0,units,NaN,NaN,0.2,NaN,NaN
4,CZE,Czech Republic,HO_BASE,Dwellings without basic facilities,L,Value,TOT,Total,PC,Percentage,0,units,NaN,NaN,0.9,NaN,NaN


In [2]:
gdp_per_capita.head()

,Country,Subject Descriptor,Units,Scale,Country/Series-specific Notes,2015,Estimates Start After
0,Afghanistan,"Gross domestic product per capita, current prices",U.S. dollars,Units,"See notes for: Gross domestic product, curren...",599.994,2013.0
1,Albania,"Gross domestic product per capita, current prices",U.S. dollars,Units,"See notes for: Gross domestic product, curren...",3995.383,2010.0
2,Algeria,"Gross domestic product per capita, current prices",U.S. dollars,Units,"See notes for: Gross domestic product, curren...",4318.135,2014.0
3,Angola,"Gross domestic product per capita, current prices",U.S. dollars,Units,"See notes for: Gross domestic product, curren...",4100.315,2014.0
4,Antigua and Barbuda,"Gross domestic product per capita, current prices",U.S. dollars,Units,"See notes for: Gross domestic product, curren...",14414.302,2011.0


**Your turn:** which column in each table will we join on? (Look for the shared one.)

## 2. Prepare the data
The book says `prepare_country_stats` does this step but hides the code. Here it is: it keeps the overall rows, reshapes the happiness table to one row per country, joins it with GDP, and keeps two columns.

In [ ]:
def prepare_country_stats(oecd_bli, gdp_per_capita):
    oecd_bli = oecd_bli[oecd_bli["INEQUALITY"] == "TOT"]
    oecd_bli = oecd_bli.pivot(index="Country", columns="Indicator", values="Value")
    gdp_per_capita = gdp_per_capita.rename(columns={"2015": "GDP per capita"})
    gdp_per_capita = gdp_per_capita.set_index("Country")
    full = pd.merge(left=oecd_bli, right=gdp_per_capita, left_index=True, right_index=True)
    full = full.sort_values(by="GDP per capita")
    # the book drops a few countries by position to match its figures
    remove_indices = [0, 1, 6, 8, 33, 34, 35]
    keep_indices = list(set(range(36)) - set(remove_indices))
    return full[["GDP per capita", "Life satisfaction"]].iloc[keep_indices]

country_stats = prepare_country_stats(oecd_bli, gdp_per_capita)
country_stats.head()

## 3. Look at it
Always plot first. Does a straight line look reasonable?

In [ ]:
country_stats.plot(kind="scatter", x="GDP per capita", y="Life satisfaction")
plt.show()

## 4. Train the model and predict
`fit` finds the best line (the intercept and slope are the **parameters**). `predict` uses it on a new input.

In [ ]:
X = np.c_[country_stats["GDP per capita"]]   # feature
y = np.c_[country_stats["Life satisfaction"]]  # label

model = sklearn.linear_model.LinearRegression()
model.fit(X, y)

print("intercept:", model.intercept_, " slope:", model.coef_)
print("Cyprus prediction:", model.predict([[22587]]))   # book says about 5.96

**Your turn:** write in your notes: what do intercept and slope mean here? (Hint: the book's formula is life satisfaction = intercept + GDP * slope.)

## 5. Experiments (this is where you learn)
Try each one, then write one sentence on what changed.
1. Predict for a poorer and a richer country: change `22587` to `10000`, then `60000`.
2. Draw the fitted line on the scatterplot (next cell).
3. Swap the model for k-nearest neighbors (the book does this next) and compare predictions.

In [ ]:
plt.scatter(X, y)
xs = np.linspace(X.min(), X.max(), 100).reshape(-1, 1)
plt.plot(xs, model.predict(xs), color="red")
plt.xlabel("GDP per capita"); plt.ylabel("Life satisfaction")
plt.show()

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

knn = KNeighborsRegressor(n_neighbors=3)
knn.fit(X, y)
print("KNN prediction for Cyprus:", knn.predict([[22587]]))

**Your turn:** the book calls this *model-based vs instance-based* learning. Which one is linear regression, and which is KNN? Look up the difference in Chapter 1 and write it in your own words.